# Data Processing Pipelines in Practice
Accompanies **Chapter 5: Data Processing Pipelines in Practice**.

Mathematical algorithms require design matrices $\mathbf{X} \in \mathbb{R}^{n \times p}$ where every entry is a finite real number. 
Raw business tables violate this requirement: numerical measurements are formatted as strings with symbols (`$`, `,`, `%`), missing values are omitted or tokenized arbitrarily, and categorical levels carry no native ordering.

This notebook builds a production-grade, leak-free preprocessing pipeline on the peer-to-peer loan dataset (`loan_data_train.csv`) matching the transformer architecture discussed in Chapter 5.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Connect to custom transformers in utils/ml_utils.py
sys.path.append(os.path.abspath('../../utils'))
from ml_utils import VarNumericCleaner, VarImputer, VarCategoricalEncoder, DataFramePipeline

data_path = os.path.abspath('../../data/loan_data_train.csv')
df = pd.read_csv(data_path)
df.shape

### Auditing Storage Types vs. Information Types
In pandas, string formatting forces columns into the `object` dtype. 
Applying the **Difference Test** from Chapter 5:
- `Interest.Rate`: A difference of $15\% - 10\% = 5\%$ represents a genuine continuous cost of capital. This is true numeric information stored as an object.
- `Debt.To.Income.Ratio`: A continuous leverage ratio stored with a trailing `%` character.

In [ ]:
print("Storage types:")
print(df[['Interest.Rate', 'Debt.To.Income.Ratio', 'Home.Ownership']].dtypes)

print("\nRaw string values:")
df[['Interest.Rate', 'Debt.To.Income.Ratio']].head(3)

### Stripping Non-Numeric Formatting and Coercing to Float
Before mathematical modeling, currency symbols, commas, and percentage characters must be stripped at the byte level. Any corrupted or unparseable string must be coerced to `np.nan` rather than raising uncaught runtime exceptions.

In [ ]:
cleaner = VarNumericCleaner(
    columns=['Interest.Rate', 'Debt.To.Income.Ratio'], 
    strip_chars=['%', '$', ',']
)
df_clean = cleaner.fit_transform(df)

df_clean[['Interest.Rate', 'Debt.To.Income.Ratio']].dtypes

### The 3-Step Categorical Encoding Strategy
Feeding arbitrary integer codes (e.g. `RENT=1, OWN=2, MORTGAGE=3`) to a linear model imposes an artificial distance metric ($3 - 1 = 2$). Instead, we apply the 3-step encoding pattern:
1. **Explicit Missing Token:** Convert unrecorded categories into an explicit class (`__missing__`) so the model can capture whether the omission itself has predictive power.
2. **Clubbing Rare Levels:** Rare categories ($< 1\%$ frequency) inflate model variance without sufficient sample support. They are collapsed into `__low_freq_cat__`.
3. **Dropping One Baseline Dummy ($k-1$ Dummies):** If an intercept column $\mathbf{1}$ is present, retaining all $k$ binary indicators causes perfect multicollinearity ($\sum_{j=1}^k D_j = \mathbf{1}$), making $\mathbf{X}^T \mathbf{X}$ singular and non-invertible. We drop one baseline level.

In [ ]:
encoder = VarCategoricalEncoder(
    columns=['Home.Ownership'],
    missing_token='__missing__',
    rare_threshold=0.01,
    drop_first=True
)
df_encoded = encoder.fit_transform(df_clean)

dummy_columns = [col for col in df_encoded.columns if col.startswith('Home.Ownership_')]
print("Generated k-1 dummy indicators:")
print(dummy_columns)
df_encoded[dummy_columns].head(3)

### Auditing Operational Feature Leakage
In `loan_data_train.csv`, the column `Amount.Funded.By.Investors` records the total capital committed after loan approval and investor listing. 
At the moment a borrower applies for a loan, this number does not exist. Including it creates severe temporal leakage, producing artificially inflated training performance that collapses completely in production.

In [ ]:
if 'Amount.Funded.By.Investors' in df_encoded.columns:
    df_encoded = df_encoded.drop(columns=['Amount.Funded.By.Investors'])
    print("Dropped 'Amount.Funded.By.Investors': post-decision feature successfully eliminated.")